# Utils

In [ ]:
from pathlib import Path
import subprocess
import os

def install_cloudflared(cloudflared_exec_path='/usr/local/bin/cloudflared'):
    cloudflared_path = Path(cloudflared_exec_path)

    if cloudflared_path.exists():
      return
    print('Installing cloudflared...')

    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

    if Path('cloudflared-linux-amd64.deb').exists():
        Path('cloudflared-linux-amd64.deb').unlink()
    print('cloudflared installed.')

def start_cloudflared(port, cloudflared_exec_path='/usr/local/bin/cloudflared'):
    cloudflared_path = Path(cloudflared_exec_path)
    print(f'Starting cloudflared tunnel for port {port}...')

    command = [
        cloudflared_path.name,
        'tunnel',
        '--url', f'http://127.0.0.1:{port}',
        ]
    print(f"Running command: {' '.join(command)}")
    process = subprocess.run(command)
    print(f"Cloudflared tunnel process started (PID: {process.pid}). Look for the public URL in its output.")
    return process # Return the process object if caller needs to manage it (e.g., terminate)


In [ ]:
install_cloudflared()
start_cloudflared(8189)

Installing cloudflared...
Selecting previously unselected package cloudflared.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared installed.
Starting cloudflared tunnel for port 8189...
Running command: cloudflared tunnel --url http://127.0.0.1:8189


KeyboardInterrupt: 

# Preparation

In [ ]:
from pathlib import Path
from google.colab import drive

# ================= CONFIGURATION =================
# Пути на Google Диске
GDRIVE_BASE     = Path('/content/drive')
COMFY_DATA_BASE = GDRIVE_BASE     / 'MyDrive/Colab Data/ComfyUI'
MODELS_DIR      = COMFY_DATA_BASE / 'models/checkpoints'
OUTPUT_DIR      = COMFY_DATA_BASE / 'output'

# Путь установки в Colab (локально для скорости)
INSTALL_BASE = Path('/content/ComfyUI')

# Флаги запуска (FP8 экономит VRAM, позволяя XL летать на T4)
ARGS = [
    "--listen 0.0.0.0",
    "--fp8_e4m3fn-text-enc",
    "--highvram",
    "--preview-method auto",
]

#MISC
CLOUDFLARED = Path('/usr/local/bin/cloudflared')
# =================================================
if not GDRIVE_BASE.is_dir():
  drive.mount(str(GDRIVE_BASE))

GDRIVE_BASE.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# =================================================
!apt-get update -qq && apt-get install -y -qq aria2

if not CLOUDFLARED.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

if not INSTALL_BASE.is_dir():
    !git clone -q https://github.com/Comfy-Org/ComfyUI.git {INSTALL_BASE}
    %cd {INSTALL_BASE}
    !pip install -r requirements.txt --quiet
else:
    %cd {INSTALL_BASE}
    !git pull -q

!rm -rf {INSTALL_BASE}/models/checkpoints && ln -s {MODELS_DIR} {INSTALL_BASE}/models
!rm -rf {INSTALL_BASE}/output && ln -s {OUTPUT_DIR} {INSTALL_BASE}/output

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/content/ComfyUI
ln: failed to create symbolic link '/content/ComfyUI/models/Colab': File exists
ln: target '/content/ComfyUI/output': No such file or directory


In [ ]:
print('Installing cloudflared...')
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess
import threading
import time

def start_comfyui():
    print('Starting ComfyUI...')
    # Make sure to run ComfyUI from its directory
    # Use --listen 0.0.0.0 to make it accessible from outside
    # Use --port 8188 as a common port, though any free port can be used
    subprocess.run(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'])

def start_cloudflared_tunnel():
    print('Starting cloudflared tunnel...')
    # Wait a bit for ComfyUI to start up
    time.sleep(10)
    # Cloudflared will create a tunnel to the specified port
    subprocess.run(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'])

# Run ComfyUI in a separate thread
comfyui_thread = threading.Thread(target=start_comfyui)
comfyui_thread.start()

# Run cloudflared tunnel in the main thread (or also in a separate thread if needed for other tasks)
start_cloudflared_tunnel()

print("ComfyUI and cloudflared tunnel started. Look for the public URL above.")

In [ ]:
import os
from google.colab import userdata
os.environ['CIVITAI_KEY'] = userdata.get('CIVITAI_API_KEY')

!aria2c --console-log-level=error \
        --summary-interval=10 \
        --file-allocation=none \
        -x 16 -s 16 -k 1M \
        -d "/content/drive/MyDrive/Colab Data/ComfyUI/models/checkpoints" \
        "https://civitai.com/api/download/models/1759168?type=Model&format=SafeTensor&size=full&fp=fp16&token=$CIVITAI_KEY"

 *** Download Progress Summary as of Sat Feb 28 04:37:18 2026 *** 
=
[#604fc3 0B/6.6GiB(0%) CN:1 DL:0B]
FILE: /content/drive/MyDrive/Colab Data/ComfyUI/models/checkpoints/juggernautXL_ragnarokBy.safetensors
-

 *** Download Progress Summary as of Sat Feb 28 04:37:28 2026 *** 
=
[#604fc3 529MiB/6.6GiB(7%) CN:16 DL:55MiB ETA:1m51s]
FILE: /content/drive/MyDrive/Colab Data/ComfyUI/models/checkpoints/juggernautXL_ragnarokBy.safetensors
-

 *** Download Progress Summary as of Sat Feb 28 04:37:39 2026 *** 
=
[#604fc3 1.0GiB/6.6GiB(16%) CN:16 DL:52MiB ETA:1m47s]
FILE: /content/drive/MyDrive/Colab Data/ComfyUI/models/checkpoints/juggernautXL_ragnarokBy.safetensors
-

 *** Download Progress Summary as of Sat Feb 28 04:37:49 2026 *** 
=
[#604fc3 1.6GiB/6.6GiB(24%) CN:16 DL:54MiB ETA:1m34s]
FILE: /content/drive/MyDrive/Colab Data/ComfyUI/models/checkpoints/juggernautXL_ragnarokBy.safetensors
-

 *** Download Progress Summary as of Sat Feb 28 04:37:59 2026 *** 
=
[#604fc3 2.3GiB/6.6GiB(34%) CN:16 D

In [ ]:
# Устанавливаем (если вдруг нет)
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download
from google.colab import userdata
from huggingface_hub import snapshot_download

# Настройки
repo_id = "Qwen/Qwen-Image-Edit-2511"
local_dir = f"/content/drive/MyDrive/Colab Data/_HF/{repo_id}"
token = userdata.get('HF_API_KEY')

# Качаем всё!
snapshot_download(
    repo_id=repo_id,
    local_dir=local_dir,
    token=token,
    endpoint=None, # Можно оставить None
    ignore_patterns=["*.msgpack", "*.h5"] # Лайфхак: игнорим ненужные форматы, чтобы сэкономить место
)

Fetching 35 files:   0%|          | 0/35 [00:00<?, ?it/s]

'/content/drive/MyDrive/Colab Data/_HF/Qwen/Qwen-Image-Edit-2511'